# Add or Release Job Holds #

This sample will create a new Workflow item and some sample jobs that will be used to showcase how to add and release holds. 

### Make connections ###

In [2]:
import arcgis
import re
import datetime
from arcgis.gis.workflowmanager import WorkflowManager

gis = arcgis.gis.GIS(url='https://organizationUrl/portal', username='admin', password='...')
item = gis.content.search('title:"Python Sample"')[0]
workflowManager = WorkflowManager(item)
print("Created connection to workflow manager item")

Created connection to workflow manager item


### Create a Job ###

In [21]:
# Create Job 
job_templates = workflowManager.job_templates
job_template = {}
for x in job_templates:
    if x.job_template_name == 'Introduction to Workflow Manager':
        job_template = x

jobs = workflowManager.jobs.create(template=job_template.job_template_id,
                                    count=2,
                                    name='Test New Job123',
                                    start='2020-04-02T13:25:50Z',
                                    end='2020-04-02T13:25:50Z',
                                    priority='High',
                                    description='job description',
                                    owner='admin',
                                    assigned='admin',
                                    complete=42,
                                    notes='testing notes'
                                    )
job_one_id = jobs[0]
job_two_id = jobs[1]
print('Ids: ' + job_one_id + ' , ' + job_two_id)


Ids: naNYvuMnRyi-uGxM_nb2Ng , AL41je6PTsaWZnkhI7hy-Q


### Add a Simple Job Hold

In [22]:
job_one = workflowManager.jobs.get(job_one_id)
job_one_diagram = workflowManager.jobs.diagram(job_one_id)
job_one_step_id = job_one_diagram.initial_step_id

result = job_one.add_hold(step_ids=[job_one_step_id])

# Refresh Job information
job_one = workflowManager.jobs.get(job_one_id)

print(result)
print(job_one.holds)

True
[{'jobId': 'naNYvuMnRyi-uGxM_nb2Ng', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'holdId': 'i_qwqUGcTRWKz_JZO98Cbg', 'setBy': 'admin', 'setDate': '2023-12-21T20:39:42Z'}]


### Release Simple Hold

In [23]:
result = job_one.release_hold(step_ids=[job_one_step_id])

# Refresh Job information
job_one = workflowManager.jobs.get(job_one_id)

print(result)
print(job_one.holds)

True
[{'jobId': 'naNYvuMnRyi-uGxM_nb2Ng', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'holdId': 'i_qwqUGcTRWKz_JZO98Cbg', 'setBy': 'admin', 'setDate': '2023-12-21T20:39:42Z', 'releasedBy': 'admin', 'releasedDate': '2023-12-21T20:39:45Z'}]


### Add a Dependent Job Hold 

In [24]:
job_two = workflowManager.jobs.get(job_two_id)
job_two_diagram = workflowManager.jobs.diagram(job_two_id)
job_two_step_id = job_two_diagram.steps[1]['id']

# Add a hold to "job_one" dependent on the next step of "job_two"
result = job_one.add_hold(step_ids=[job_one_step_id], dependent_job_id=job_two_id, dependent_step_id= job_two_step_id)

# Refresh Job information
job_one = workflowManager.jobs.get(job_one_id)

print(result)
print(job_one.holds[0])
print(job_one.holds[1])

True
{'jobId': 'naNYvuMnRyi-uGxM_nb2Ng', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'holdId': 'i_qwqUGcTRWKz_JZO98Cbg', 'setBy': 'admin', 'setDate': '2023-12-21T20:39:42Z', 'releasedBy': 'admin', 'releasedDate': '2023-12-21T20:39:45Z'}
{'jobId': 'naNYvuMnRyi-uGxM_nb2Ng', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'holdId': 'wp0YYP2rTX-tL9iau0Z-rQ', 'setBy': 'admin', 'setDate': '2023-12-21T20:39:49Z', 'dependentJobId': 'AL41je6PTsaWZnkhI7hy-Q', 'dependentStepId': '5ab308af-6961-a654-1247-6b7cb9a65268'}


### Release a Dependent Job Hold

In [25]:
result = job_one.release_hold(step_ids=[job_one_step_id], dependent_job_id=job_two_id, dependent_step_id= job_two_step_id)

# Refresh Job information
job_one = workflowManager.jobs.get(job_one_id)

print(result)
print(job_one.holds[0])
print(job_one.holds[1])

True
{'jobId': 'naNYvuMnRyi-uGxM_nb2Ng', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'holdId': 'i_qwqUGcTRWKz_JZO98Cbg', 'setBy': 'admin', 'setDate': '2023-12-21T20:39:42Z', 'releasedBy': 'admin', 'releasedDate': '2023-12-21T20:39:45Z'}
{'jobId': 'naNYvuMnRyi-uGxM_nb2Ng', 'stepId': 'df4c8d20-5c99-457f-0be1-21fa8f830760', 'holdId': 'wp0YYP2rTX-tL9iau0Z-rQ', 'setBy': 'admin', 'setDate': '2023-12-21T20:39:49Z', 'releasedBy': 'admin', 'releasedDate': '2023-12-21T20:39:57Z', 'dependentJobId': 'AL41je6PTsaWZnkhI7hy-Q', 'dependentStepId': '5ab308af-6961-a654-1247-6b7cb9a65268'}
